# 个人量化 · 零成本云端训练（Kaggle Notebook 模板）

一键跑完 **数据采集 → LightGBM Walk-Forward 训练 → 导出轻量模型 JSON**，产物 `model/latest_model.json` 提交回仓库供 Cloudflare Worker 推理。

- ✅ **Kaggle 免费 T4×2（约 30 小时/周）**：本 Notebook 首选；LightGBM PyPI 版是 CPU 版，**不需要 GPU**，CPU 训练 1 分钟级
- ✅ 需要 **开启 Internet**（右上角 Settings → Internet → On），以及 `requirements.txt` 里的免费数据源（AKShare 首选 + Baostock 兜底）
- ✅ 全流程约 10–20 分钟；脚本幂等增量，重复运行不会重复下载

> ⚠️ 仅供学习研究，非投资建议。

In [ ]:
import os, subprocess, sys

REPO = "/kaggle/working/personal-quant"
REPO_URL = "https://github.com/xiongkong520/personal-quant.git"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
print("工作目录:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 安装训练依赖（Kaggle 首次运行约 1 分钟；后续可注释掉）
!pip install -q -r training/requirements.txt
import akshare, baostock, lightgbm
print("akshare", akshare.__version__, "| lightgbm", lightgbm.__version__)

## 第 1 步：数据采集（近 3 年前复权日线 + 最近交易日涨跌停池）

- 默认 40 只股票池，`--symbols` 可改小样本；
- AKShare 首选，被东财限流时自动切 Baostock（免费官方兜底）；
- 输出 `training/data/{daily,zt_pool,index}.parquet`（不提交仓库）。

> 涨跌停池免费接口只覆盖最近约 10~30 个交易日，属正常限制；历史情绪由训练脚本用股票池代理特征补齐。

In [ ]:
# 全量 40 只池（约 10-20 分钟）；小样本可改成 --symbols 600519,000001 --start 2024-01-01
!python training/fetch_data.py --start 2023-11-01 --zt-days 30

## 第 2 步：特征 + LightGBM 三分类 Walk-Forward

- 30 个特征，全部 `<=t` 数据 + 市场情绪 `shift(1)` 防未来函数；
- 标签：T+1 次日开盘买/收盘卖，`|r| <= 0.2%`（双边摩擦）记 flat；
- 训练窗 250 交易日滚动、验证窗 20 交易日，输出样本外 accuracy / win_rate / logloss / Wilson CI。

In [ ]:
!python training/train_lgbm.py --train-days 250 --valid-days 20 --step 20 --num-iterations 120 --num-leaves 15

## 第 3 步：导出轻量模型 JSON（`model/latest_model.json`，目标 < 2MB）

In [ ]:
!python training/export_model.py --max-size-mb 2.0
import json
m = json.load(open("model/latest_model.json"))
print("schema:", m["schema"], "| model:", m["model"], "| version:", m["version"])
print("trees:", len(m["trees"]), "| features:", len(m["features"]), "| classes:", m["label"]["classes"])
print("walk_forward:", json.dumps(m["metrics"]["walk_forward"], ensure_ascii=False)[:400])

In [ ]:
# 产物大小与校验（如 > 2MB 会由导出脚本自动裁剪树数并重算指标）
import os, json
for p in ("model/latest_model.json", "model/feature_config.json"):
    print(p, round(os.path.getsize(p)/1024, 1), "KB")
from training.export_model import validate_model_json  # noqa
print("schema 校验:", validate_model_json(json.load(open("model/latest_model.json"))) or "OK")

## 下载并提交模型

1. 右侧 **Output** 面板 → 下载 `personal-quant/model/latest_model.json` 与 `feature_config.json`；
2. 覆盖仓库 `model/` 下同名文件，提交 PR / push；
3. GitHub Actions 每天定时重训时也会自动更新这两个文件（由 Lead 接入 workflow）。

### 备选方案
- **Colab 免费**：同步骤，运行时选默认 CPU 即可；中断后重跑幂等增量；
- **GitHub Actions 兜底**：2 vCPU 也够，40 只池全流程 < 10 分钟；`--zt-days 1` 只抓当日情绪池，历史靠每日累积。

In [ ]:
# 可选：把产物打包，方便一次性下载
import shutil, os
shutil.make_archive("/kaggle/working/model_export", "zip", "model")
print("已打包:", os.path.getsize("/kaggle/working/model_export.zip"), "bytes")